In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import silhouette_score
from sklearn.neighbors import KNeighborsClassifier


In [10]:
zbior = load_breast_cancer(as_frame=True)

X = zbior.data
y = zbior.target  #0 rak zlosliwy, 1 rak łagodny

print("Nazwy klas:")
for numer, nazwa in enumerate(zbior.target_names):
    print(numer, "=", nazwa)


print("Pierwsze wiersze cech:")
print(X.head())


print("Liczba obserwacji w klasach:")
print(y.value_counts().sort_index())

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)


print("Rozmiar X_train:", X_train.shape)
print("Rozmiar X_test:", X_test.shape)


model = Pipeline(steps=[
    ("skalowanie", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)


accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", round(accuracy, 3))

prawdopodobienstwa = model.predict_proba(X_test)

proba_klasa_0 = prawdopodobienstwa[:, 0]
proba_klasa_1 = prawdopodobienstwa[:, 1]

wyniki = pd.DataFrame({
    "klasa_prawdziwa": y_test.values,
    "P(klasa=0)": proba_klasa_0,
    "P(klasa=1)": proba_klasa_1,
    "klasa_predict": y_pred,
})



mapa_klas = {
    0: zbior.target_names[0],
    1: zbior.target_names[1],
}

wyniki["prawdziwa_nazwa"] = wyniki["klasa_prawdziwa"].map(mapa_klas)
wyniki["przewidziana_nazwa"] = wyniki["klasa_predict"].map(mapa_klas)


print("Pierwsze 10 predykcji i prawdopodobieństw:")
print(wyniki.head(10))



y_pred_prog_05 = (proba_klasa_1 >= 0.5).astype(int)

print("Czy ręczny próg 0.5 daje te same klasy co predict()?")
print(np.array_equal(y_pred, y_pred_prog_05))

y_pred_prog_03 = (proba_klasa_1 >= 0.3).astype(int)
y_pred_prog_04 = (proba_klasa_1 >= 0.4).astype(int)
y_pred_prog_07 = (proba_klasa_1 >= 0.7).astype(int)

print("Liczba przewidzianej klasy 1 dla różnych progów:")
print("Próg 0.3:", int(y_pred_prog_03.sum()))
print("Próg 0.4:", int(y_pred_prog_04.sum()))
print("Próg 0.5:", int(y_pred_prog_05.sum()))
print("Próg 0.7:", int(y_pred_prog_07.sum()))

accuracy_03 = accuracy_score(y_test, y_pred_prog_03)
accuracy_07 = accuracy_score(y_test, y_pred_prog_07)

print("Accuracy_dla progu 0.3:", round(accuracy_03, 3))
print("Accuracy_dla progu 0.3:", round(accuracy_07, 3))

regresja_logistyczna = model.named_steps["model"]

wspolczynniki = pd.Series(
    regresja_logistyczna.coef_[0],
    index=X.columns,
)

print("Największe dodatnie współczynniki:")
print(wspolczynniki.sort_values(ascending=False).head(8))


print("Najbardziej ujemne współczynniki:")
print(wspolczynniki.sort_values().head(8))

iloraz_szans = np.exp(wspolczynniki)
print("Przykładowe największe ilorazy szans:")
print(iloraz_szans.sort_values(ascending=False).head(8))

print("Wyraz wolny:")
print(round(regresja_logistyczna.intercept_[0], 3))


print("Interpretacja:")
print("1. Regresja logistyczna jest modelem klasyfikacyjnym, mimo słowa 'regresja' w nazwie.")
print("2. predict() zwarca gotową klasę, natomiast predict_proba() zwraca prawdopodobieństwo klas.")
print("3. Sam współczynnik nie oznacza automatycznie związku przyczynowego między cechą na klasą.")

Nazwy klas:
0 = malignant
1 = benign
Pierwsze wiersze cech:
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  